> (중요) 여러분 구글 드라이브에 최소 7GB 이상은 확보되어 있어야 합니다!

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import unicodedata  # 0번 섹션에 추가 필요

# 📌 경로 설정 (제공해주신 경로 반영)
def normalize_path(path):
    # 1. unicodedata.normalize('NFC', path): 경로 문자열을 NFC 방식으로 통일
    # 2. .strip(): 앞뒤에 붙은 불필요한 공백 제거
    return unicodedata.normalize('NFC', path).strip()

> (주의) 아래 코드는 처음 딱 한 번만!

In [ ]:
import zipfile
import os
import shutil
import time

# 1. 경로 설정
dataset_zip = normalize_path("/content/drive/MyDrive/data/초급_프로젝트/초급_프로젝트_수강생_배포용.zip")
extract_path = normalize_path("/content/drive/MyDrive/data/초급_프로젝트/dataset/")

os.makedirs(extract_path, exist_ok=True)

# 2. 메인 압축파일 해제
print(f"📦 메인 데이터셋 해제 중: {os.path.basename(dataset_zip)}")
with zipfile.ZipFile(dataset_zip, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

# 메인 압축파일 삭제 (요청사항)
os.remove(dataset_zip)
print("🗑️ 메인 압축파일 삭제 완료.")

# 3. 내부 이미지 압축파일 통합 해제 로직
print("\n🚀 이미지 폴더 통합 및 내부 압축 해제 시작...")
for file in os.listdir(extract_path):
    if file.endswith(".zip"):
        file_path = os.path.join(extract_path, file)
        
        # 이름 기반 대상 폴더 결정 (train/test 통합)
        if 'train' in file.lower():
            target_folder_name = "train_images"
        elif 'test' in file.lower():
            target_folder_name = "test_images"
        else:
            target_folder_name = file.replace(".zip", "")

        target_subfolder = os.path.join(extract_path, target_folder_name)
        os.makedirs(target_subfolder, exist_ok=True)

        print(f"📂 {file} -> {target_folder_name} 통합 중...")
        
        with zipfile.ZipFile(file_path, 'r') as zip_ref:
            for member in zip_ref.infolist():
                if not member.is_dir():
                    # 내부 경로 구조를 무시하고 파일명만 추출하여 저장
                    filename = os.path.basename(member.filename)
                    if filename:
                        target_file_path = os.path.join(target_subfolder, filename)
                        with zip_ref.open(member) as source, open(target_file_path, "wb") as target:
                            shutil.copyfileobj(source, target)
        
        # [수정 포인트] 삭제 전 잠시 대기 후 강제 삭제 시도
        try:
            time.sleep(0.5) 
            if os.path.exists(file_path):
                os.remove(file_path)
                print(f"🗑️ 삭제 성공: {file}")
        except Exception as e:
            print(f"❌ {file} 삭제 실패: {e}")

print("\n✨ 모든 작업이 완료되었습니다!")
print(f"📁 최종 데이터셋 구성: {os.listdir(extract_path)}")

- 구글 드라이브 휴지통 비우기

In [ ]:
from google.colab import auth
from googleapiclient.discovery import build

# 1. 구글 드라이브 인증
auth.authenticate_user()
drive_service = build('drive', 'v3')

# 2. 휴지통 완전히 비우기 함수
def empty_trash():
    try:
        drive_service.files().emptyTrash().execute()
        print("✅ 구글 드라이브 휴지통이 완전히 비워졌습니다.")
    except Exception as e:
        print(f"❌ 휴지통 비우기 실패: {e}")

# 실행
empty_trash()

> 압축 해제한 파일들의 반영 시간이 걸릴 수 있으므로, 커널 재시작 해주기!

#### `normalize_path`는?

`normalize_path`는 파일 경로에 포함된 **한글(유니코드) 처리 방식**을 통일하여, 경로를 찾지 못하는 에러를 방지하기 위한 함수입니다.

특히 **Google Colab**이나 **Mac, Windows** 사이에서 데이터를 주고받을 때 한글 폴더명이 깨져서 발생하는 `File Not Found` 에러를 잡는 데 필수적입니다.

<br>

##### 왜 사용하나요? (NFC vs NFD)

한글을 컴퓨터가 인식하는 방식은 크게 두 가지입니다.

* **NFC (Windows 스타일):** '강'을 '강'이라는 하나의 글자로 저장합니다.
* **NFD (Mac/Unix 스타일):** '강'을 'ㄱ', 'ㅏ', 'ㅇ'으로 쪼개서 저장합니다.

사람 눈에는 똑같이 "초급 프로젝트"라고 보이지만, 컴퓨터 입장에서는 글자 조합 방식이 다르면 **완전히 다른 경로**로 인식합니다. `normalize_path` 는 이를 **NFC(표준 방식)** 로 강제 통일해주는 역할을 합니다.

In [ ]:
############################################################
# 0. 라이브러리 임포트 & 경로 설정
############################################################
import os
import json
import pandas as pd
from PIL import Image
import unicodedata  # 0번 섹션에 추가 필요
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import torchvision
import torchvision.transforms as T
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

# 📌 경로 설정 (제공해주신 경로 반영)
def normalize_path(path):
    # 1. unicodedata.normalize('NFC', path): 경로 문자열을 NFC 방식으로 통일
    # 2. .strip(): 앞뒤에 붙은 불필요한 공백 제거
    return unicodedata.normalize('NFC', path).strip()

# 경로는 환경에 맞게 수정
# train_images, test_images
extract_path = normalize_path("/content/drive/MyDrive/data/초급_프로젝트/dataset/") # 압축을 풀 폴더

TRAIN_JSON_PATH = os.path.join(extract_path, "merged_annotations_train_final.json")
TEST_JSON_PATH = os.path.join(extract_path, "merged_annotations_test_final.json")
TRAIN_IMG_DIR = os.path.join(extract_path, "train_images")
TEST_IMG_DIR  = os.path.join(extract_path, "test_images")

# merged_annotation json 경로
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

############################################################
# 1. 병합된 JSON 파일을 읽어서 DataFrame으로 만들기
############################################################

def build_df_from_merged_json(json_path, img_dir):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # 1) 이미지 정보 매핑 (id -> file_name)
    id_to_fname = {img["id"]: img["file_name"] for img in data["images"]}

    records = []
    # 2) 어노테이션 순회
    for ann in data["annotations"]:
        img_id_coco = ann["image_id"]
        file_name = id_to_fname.get(img_id_coco)
        
        if file_name is None: continue
        
        img_path = os.path.join(img_dir, file_name)
        
        # 실제 이미지 파일이 있는지 확인 (선택 사항이지만 안전함)
        if not os.path.exists(img_path):
            continue

        x, y, w, h = ann["bbox"]
        
        records.append({
            "image_path": img_path,
            "image_id": os.path.splitext(file_name)[0], # 파일명을 ID로 사용
            "category_id": int(ann["category_id"]),
            "bbox_x": float(x),
            "bbox_y": float(y),
            "bbox_w": float(w),
            "bbox_h": float(h),
        })

    return pd.DataFrame(records)

# 실행
df = build_df_from_merged_json(TRAIN_JSON_PATH, TRAIN_IMG_DIR)
print(f"✅ 학습 데이터 로드 완료: {len(df)} 개의 객체 탐지됨")

In [ ]:
############################################################
# 2. category_id 매핑 (겉으로는 안 바꾸고, 모델 내부에서만 사용)
############################################################

# 원본 category_id 집합
unique_cats = sorted(df["category_id"].unique())
print("고유 category_id 개수:", len(unique_cats))

# 내부용: 모델에 넣을 label (1 ~ num_classes-1), 0은 background
orig2model = {cid: i + 1 for i, cid in enumerate(unique_cats)}   # 원본 → 모델용
model2orig = {v: k for k, v in orig2model.items()}               # 모델용 → 원본

num_classes = len(unique_cats) + 1  # background 포함
print("num_classes (background 포함):", num_classes)

In [ ]:
############################################################
# 3. Dataset 정의
############################################################

class OralDrugDataset(Dataset):
    """
    Faster R-CNN용 Dataset
    - __getitem__은 (image, target) 반환
    - image: [C,H,W] float tensor
    - target: dict(boxes, labels, image_id, ...)
    """
    def __init__(self, df, orig2model, transforms=None):
        self.df = df.reset_index(drop=True)
        self.orig2model = orig2model
        self.transforms = transforms

        # 이미지 단위로 그룹을 만들기 위해 고유 image_id 리스트를 유지
        self.image_ids = self.df["image_id"].unique().tolist()

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        # 1) image_id 선택
        image_id = self.image_ids[idx]

        # 2) 해당 image_id에 해당하는 모든 row (여러 객체) 가져오기
        df_img = self.df[self.df["image_id"] == image_id]

        # 3) 이미지 로드
        img_path = df_img["image_path"].iloc[0]
        image = Image.open(img_path).convert("RGB")

        # 4) bbox / labels 구성
        boxes = []
        labels = []

        for _, row in df_img.iterrows():
            x = row["bbox_x"]
            y = row["bbox_y"]
            w = row["bbox_w"]
            h = row["bbox_h"]

            # [x1, y1, x2, y2] 형식으로 변환
            x1 = x
            y1 = y
            x2 = x + w
            y2 = y + h
            boxes.append([x1, y1, x2, y2])

            # 원본 category_id → 모델용 label로 변환
            orig_cat = int(row["category_id"])
            model_cat = self.orig2model[orig_cat]
            labels.append(model_cat)

        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            # image_id는 loss에는 크게 영향 없음, 그냥 인덱스 정도로 넣어도 무방
            "image_id": torch.tensor([idx]),
        }

        if self.transforms is not None:
            image, target = self.transforms(image, target)

        return image, target


############################################################
# 4. 데이터 증강 및 DataLoader 구성
############################################################

class PillDetectionTransforms:
    def __init__(self, is_train=True):
        self.is_train = is_train

    def __call__(self, image, target):
        # 1. 훈련 모드일 때만 랜덤 증강 적용
        if self.is_train:
            # 좌우 반전 (50% 확률)
            if torch.rand(1) > 0.5:
                image = T.functional.hflip(image)
                width, _ = image.size
                boxes = target["boxes"]
                # x1, x2 좌표 반전: [width - x2, y1, width - x1, y2]
                boxes[:, [0, 2]] = width - boxes[:, [2, 0]]
                target["boxes"] = boxes

            # 상하 반전 (50% 확률)
            if torch.rand(1) > 0.5:
                image = T.functional.vflip(image)
                _, height = image.size
                boxes = target["boxes"]
                # y1, y2 좌표 반전: [x1, height - y2, x2, height - y1]
                boxes[:, [1, 3]] = height - boxes[:, [3, 1]]
                target["boxes"] = boxes

            # 밝기, 대비, 채도 조절
            if torch.rand(1) > 0.3: # 70% 확률로 색감 변화
                color_jitter = T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1)
                image = color_jitter(image)

        # 2. 공통: 텐서 변환
        image = T.functional.to_tensor(image)
        return image, target


# Dataset 객체 생성
train_dataset_full = OralDrugDataset(df, orig2model=orig2model, transforms=PillDetectionTransforms(is_train=True))
val_dataset_full = OralDrugDataset(df, orig2model=orig2model, transforms=PillDetectionTransforms(is_train=False))

# 데이터 분할
indices = torch.randperm(len(train_dataset_full)).tolist()
split = int(0.9 * len(indices))
train_indices = indices[:split]
val_indices = indices[split:]

train_dataset = torch.utils.data.Subset(train_dataset_full, train_indices)
val_dataset = torch.utils.data.Subset(val_dataset_full, val_indices)

def collate_fn(batch):
    return tuple(zip(*batch))

train_loader = DataLoader(
    train_dataset,
    batch_size=2, 
    shuffle=True,
    collate_fn=collate_fn,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=collate_fn,
)

print(f"✅ 증강 완료! 학습용: {len(train_dataset)}건, 검증용: {len(val_dataset)}건")



* 모델이 다양한 환경을 학습할 수 있도록 데이터 증강
* 좌우/상하 반전 : 알약의 방향에 상관없이 이미지 인식
* 색감 조절 : 조명 조건에 따른 알약 색상 변화로 발생할 수 있는 오류 보완
* 정확한 좌표 동기화 : 모델이 잘못된 위치를 학습하지 않도록 보완

In [ ]:

############################################################
# 5. 모델 정의 (Faster R-CNN + ResNet50 FPN 전이학습)
############################################################

# 사전학습된 Faster R-CNN 모델 로드
model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")

# 분류 head를 우리 데이터셋 클래스 개수에 맞게 교체
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

model.to(DEVICE)

# 옵티마이저 설정
params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.AdamW(params, lr=1e-4, weight_decay=1e-4)

# (선택) 러닝레이트 스케줄러
lr_scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)


############################################################
# 6. 학습 루프
############################################################

num_epochs = 15  # 혹은 원하는 에폭 수
optimizer = optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

for epoch in range(num_epochs):
    ##########################
    # 1) Train
    ##########################
    model.train()
    train_loss_sum = 0.0

    for images, targets in train_loader:
        images = [img.to(DEVICE) for img in images]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        train_loss_sum += losses.item()

    avg_train_loss = train_loss_sum / max(1, len(train_loader))

    ##########################
    # 2) Validation (loss만 측정)
    ##########################
    model.train()  # ★ 여기 중요: eval()이 아니라 train() 상태에서 호출해야 loss 나옴
    val_loss_sum = 0.0
    with torch.no_grad():
        for images, targets in val_loader:
            images = [img.to(DEVICE) for img in images]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            val_loss_sum += losses.item()

    avg_val_loss = val_loss_sum / max(1, len(val_loader))

    print(f"[Epoch {epoch+1}/{num_epochs}] "
          f"train_loss: {avg_train_loss:.4f} | val_loss: {avg_val_loss:.4f}")

    # 스케줄러 한 스텝
    scheduler.step()

# 학습된 모델 저장 (원하면)
torch.save(model.state_dict(), "fasterrcnn_oral_drug.pth")
print("모델 저장 완료")



In [ ]:

############################################################
# 7. test_images에 대해 예측 → submission.csv 생성
############################################################

# test 이미지 파일 목록 가져오기
test_files = [f for f in os.listdir(TEST_IMG_DIR) if f.endswith(".png")]
test_files = sorted(test_files)

model.eval()

rows = []
annotation_id = 1      # submission용 annotation_id 시작
score_threshold = 0.3  # 너무 낮은 점수는 제거 (필요에 따라 조정)

with torch.no_grad():
    for f in test_files:
        img_path = os.path.join(TEST_IMG_DIR, f)
        image = Image.open(img_path).convert("RGB")

        # image_id = 파일명에서 확장자 제거한 문자열 그대로 사용
        image_id = os.path.splitext(f)[0]

        img_tensor = T.ToTensor()(image).to(DEVICE)
        outputs = model([img_tensor])[0]

        keep = outputs["scores"].cpu() >= score_threshold
        boxes  = outputs["boxes"].cpu()[keep]
        labels = outputs["labels"].cpu()[keep]
        scores = outputs["scores"].cpu()[keep]

        for box, lab, sc in zip(boxes, labels, scores):
            x1, y1, x2, y2 = box.tolist()
            w = x2 - x1
            h = y2 - y1

            orig_cat = model2orig[int(lab.item())]

            rows.append({
                "annotation_id": annotation_id,
                "image_id": image_id,  # 문자열 그대로 사용
                "category_id": orig_cat,
                "bbox_x": x1,
                "bbox_y": y1,
                "bbox_w": w,
                "bbox_h": h,
                "score": float(sc.item()),
            })

            annotation_id += 1

# DataFrame으로 만들고 저장
df_sub = pd.DataFrame(rows, columns=[
    "image_id", "category_id",
    "bbox_x", "bbox_y", "bbox_w", "bbox_h", "score"
])
# 이미지 ID별로 점수 높은 순 정렬 후 상위 4개 추출
df_sub = df_sub.sort_values(by=["image_id", "score"], ascending=[True, False])
df_sub = df_sub.groupby("image_id").head(4)

# 3) 최종 annotation_id 부여 (1부터 순차적으로)
df_sub.insert(0, "annotation_id", range(1, len(df_sub) + 1))

# 4) CSV 저장
output_path = "final_submission.csv"
df_sub.to_csv(os.path.join(extract_path, output_path), index=False)

print(f"✅ 생성 완료: {output_path}")
print(f"📊 총 예측 객체 수: {len(df_sub)}")
print(df_sub.head())

In [ ]:
############################################################
# 8. 모델 성능 평가 (mAP 측정)
############################################################

import json
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# 1. df_sub(Pandas)를 COCO 평가용 리스트로 변환
eval_results = []
for _, row in df_sub.iterrows():
    eval_results.append({
        "image_id": int(row["image_id"]),
        "category_id": int(row["category_id"]),
        "bbox": [row["bbox_x"], row["bbox_y"], row["bbox_w"], row["bbox_h"]],
        "score": float(row["score"])
    })

# 2. 임시 JSON 파일로 저장
temp_json = "temp_eval.json"
with open(temp_json, "w") as f:
    json.dump(eval_results, f)

# 3. COCO 평가 실행
coco_gt = COCO(TEST_JSON_PATH)
coco_dt = coco_gt.loadRes(temp_json)

coco_eval = COCOeval(coco_gt, coco_dt, "bbox")
coco_eval.evaluate()
coco_eval.accumulate()
coco_eval.summarize()